In [ ]:
## This is a script to uodate the alcohol_kansspelen column in the database,
## because sometimes this wasnt calculated properly

In [1]:
import src.utils.database_utils as db_utils
import src.utils.dictionaries as mydict
import src.utils.connection_utils as con_utils

import pandas as pd
import time
from tqdm import tqdm
import requests
import json
import ast
import numpy as np
from sqlalchemy import text

In [3]:
from sqlalchemy import text

def update_alcohol_label(flights, spots):
    for index, flight in flights.iterrows():
        # Filter spots for the current flight's advertiser, product, and date range
        flight_spots = spots[
            (spots['advertiser'] == flight['advertiser']) &
            (spots['product'] == flight['product']) &
            (spots['date'] >= flight['start_date']) &
            (spots['date'] <= flight['end_date'])
        ]

        # Check if all rows for alcohol_kansspelen are True (1)
        if len(flight_spots) > 0 and flight_spots['alcohol_kansspelen'].all():
            flights.at[index, 'alcohol_kansspelen'] = 1  # Mark as True/1
        else:
            flights.at[index, 'alcohol_kansspelen'] = 0  # Mark as False/0

def update_alcohol_column(flights_df, engine):
    # Ensure alcohol_kansspelen is converted to int (0 or 1) to match SQL BIT type
    flights_df['alcohol_kansspelen'] = flights_df['alcohol_kansspelen'].astype(int)

    update_query = """
    UPDATE [dbo].[tv.flights_ster]
    SET alcohol_kansspelen = :alcohol_kansspelen
    WHERE LTRIM(RTRIM(flight_id)) = :flight_id
    """

    with engine.begin() as conn:
        for index, row in flights_df.iterrows():
            # Execute the update query for each row
            conn.execute(
                text(update_query),
                {
                    "alcohol_kansspelen": row['alcohol_kansspelen'],
                    "flight_id": row['flight_id']
                }
            )
            # Optional: Confirm update
            print(f"Updated flight_id {row['flight_id']} with alcohol_kansspelen = {row['alcohol_kansspelen']}")

def process_and_update_by_year(start_year, end_year):
    query_spots = """
        SELECT *
        FROM [dbo].[tv.spots_ster]
        ORDER BY date, advertiser;
    """
    # Fetch all spots once, since they are reused across all years
    all_spots = db_utils.query_to_df(query_spots)

    for year in range(start_year, end_year + 1):
        print(f"Processing year: {year}")

        # Create the engine for each year to avoid timeout
        engine = con_utils.create_sql_alchemy_engine()

        # Query the flights for the current year
        query_flights = f"""
        SELECT *
        FROM [dbo].[tv.flights_ster]
        WHERE start_date >= '{year}-01-01' AND start_date <= '{year}-12-31'
        """
        flights = db_utils.query_to_df(query_flights)

        if not flights.empty:
            # Update the alcohol_kansspelen column
            update_alcohol_label(flights, all_spots)

            # Update the database for this year's flights
            update_alcohol_column(flights, engine)

        print(f"Finished processing year: {year}")

        # Dispose of the engine after use to release resources
        engine.dispose()

# Loop through years 2018 to 2024 and process each year
process_and_update_by_year(2018, 2024)


Processing year: 2018


C:\Users\jordy.schep\AppData\Local\Temp\ipykernel_4128\3790245632.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  flights.at[index, 'alcohol_kansspelen'] = 0  # Mark as False/0


Updated flight_id 4ec1da8b-5ac1-9387-3ee6-2fd57a70f3af with alcohol_kansspelen = 0
Updated flight_id 2d8fc4cf-c8cb-7372-6fd6-ccb103e1eaa5 with alcohol_kansspelen = 0
Updated flight_id 258717ca-6b89-740d-b8ae-ca85cd5e8604 with alcohol_kansspelen = 0
Updated flight_id 08eb8f36-1417-d3a0-c219-c94188e7f0eb with alcohol_kansspelen = 0
Updated flight_id 6fb95b6c-40c9-da5f-5f36-9c7d9a2c4659 with alcohol_kansspelen = 0
Updated flight_id 6b93ed16-d9a6-e2e0-0e35-60244bf2d812 with alcohol_kansspelen = 0
Updated flight_id c71354dd-9bb0-d8bf-6213-17b4a6167035 with alcohol_kansspelen = 0
Updated flight_id 94f7f338-1434-ed40-b93a-770e3147d7e4 with alcohol_kansspelen = 0
Updated flight_id 3cf9f15c-c974-41e8-8f6f-4868ca7d335c with alcohol_kansspelen = 0
Updated flight_id 72ddd10f-3c47-9df2-65ee-d53d7433efba with alcohol_kansspelen = 0
Updated flight_id 00be69a3-b36d-375d-2b55-69b89dbe80c1 with alcohol_kansspelen = 0
Updated flight_id 8413151c-2cb8-1560-ceb5-1d57924e2d78 with alcohol_kansspelen = 0
Upda

C:\Users\jordy.schep\AppData\Local\Temp\ipykernel_4128\3790245632.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  flights.at[index, 'alcohol_kansspelen'] = 0  # Mark as False/0


Updated flight_id 18eda72a-2508-eb85-630b-d078bfc5de5f with alcohol_kansspelen = 0
Updated flight_id fb932a23-0f1d-4037-39a7-0e0604ea377e with alcohol_kansspelen = 0
Updated flight_id 62b5916c-1b8b-2c9c-7547-a94fbc6860dc with alcohol_kansspelen = 0
Updated flight_id 4c0ac722-be54-c348-9c93-63e5abba8d8b with alcohol_kansspelen = 0
Updated flight_id ed876fe5-67f0-7e16-4359-51688e9ae860 with alcohol_kansspelen = 0
Updated flight_id 6ae5ee9c-3352-1eef-3d0d-b2327e5f8435 with alcohol_kansspelen = 0
Updated flight_id 7ff5983a-4152-af50-c521-33842a520a07 with alcohol_kansspelen = 0
Updated flight_id f608f606-64da-a86b-885d-9e7149e53d65 with alcohol_kansspelen = 1
Updated flight_id 4231f1a1-0dd2-5d26-3283-98bc0630b0c5 with alcohol_kansspelen = 0
Updated flight_id f3fc703b-386c-04fa-1a1e-9793c80dc5e1 with alcohol_kansspelen = 0
Updated flight_id 15e3281e-dcf7-ac34-654b-f769a72f228c with alcohol_kansspelen = 0
Updated flight_id 1b75bc1f-32ce-6f46-f378-4d397f271355 with alcohol_kansspelen = 0
Upda

C:\Users\jordy.schep\AppData\Local\Temp\ipykernel_4128\3790245632.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  flights.at[index, 'alcohol_kansspelen'] = 0  # Mark as False/0


Updated flight_id 3760a940-f7f2-bfa1-294a-49cb776491fc with alcohol_kansspelen = 0
Updated flight_id 6c2a45b2-cba6-0504-0775-296958bc528d with alcohol_kansspelen = 0
Updated flight_id 1ae56789-9207-0c89-a474-3a3c76776e7e with alcohol_kansspelen = 0
Updated flight_id e4269da7-0587-07dc-2813-b940254f5c9c with alcohol_kansspelen = 0
Updated flight_id 4dcd4bf8-3a6f-2864-16a2-0521cd6290ab with alcohol_kansspelen = 0
Updated flight_id 9cd0a3be-b333-c1f0-5bee-25a6e128510a with alcohol_kansspelen = 0
Updated flight_id 162a4d6f-a36c-f153-2a15-e53b2c37bc36 with alcohol_kansspelen = 0
Updated flight_id f3650734-279c-f124-4045-8360cab3807a with alcohol_kansspelen = 0
Updated flight_id 4dc5c866-fd4f-7fb9-ed3d-5dad533f8801 with alcohol_kansspelen = 0
Updated flight_id d347f0dc-9b0c-ceb0-9cd8-d39d17ed0114 with alcohol_kansspelen = 0
Updated flight_id 840f801d-2512-a673-7157-b016464ed762 with alcohol_kansspelen = 0
Updated flight_id d7b2805b-a5fb-73f8-3197-274902fa3c24 with alcohol_kansspelen = 0
Upda

KeyboardInterrupt: 